In [2]:
model_name = "distilgpt2"
device = "cpu"

In [3]:
!pip install transformers torch





In [4]:
!pip install transformers torch datasets pandas "transformers[torch]"

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM




tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model= AutoModelForCausalLM.from_pretrained(model_name)

model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [6]:
 !pip install kagglehub   

In [7]:
import pandas as pd 

df = pd.read_csv("wiki_movie_plots_deduped.csv")
df

,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot
0,1901,Kansas Saloon Smashers,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Kansas_Saloon_Sm...,"A bartender is working at a saloon, serving dr..."
1,1901,Love by the Light of the Moon,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Love_by_the_Ligh...,"The moon, painted with a smiling face hangs ov..."
2,1901,The Martyred Presidents,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/The_Martyred_Pre...,"The film, just over a minute long, is composed..."
3,1901,"Terrible Teddy, the Grizzly King",American,Unknown,NaN,unknown,"https://en.wikipedia.org/wiki/Terrible_Teddy,_...",Lasting just 61 seconds and consisting of two ...
4,1902,Jack and the Beanstalk,American,"George S. Fleming, Edwin S. Porter",NaN,unknown,https://en.wikipedia.org/wiki/Jack_and_the_Bea...,The earliest known adaptation of the classic f...
...,...,...,...,...,...,...,...,...
34881,2014,The Water Diviner,Turkish,Director: Russell Crowe,Director: Russell Crowe\r\nCast: Russell Crowe...,unknown,https://en.wikipedia.org/wiki/The_Water_Diviner,"The film begins in 1919, just after World War ..."
34882,2017,Çalgı Çengi İkimiz,Turkish,Selçuk Aydemir,"Ahmet Kural, Murat Cemcir",comedy,https://en.wikipedia.org/wiki/%C3%87alg%C4%B1_...,"Two musicians, Salih and Gürkan, described the..."
34883,2017,Olanlar Oldu,Turkish,Hakan Algül,"Ata Demirer, Tuvana Türkay, Ülkü Duru",comedy,https://en.wikipedia.org/wiki/Olanlar_Oldu,"Zafer, a sailor living with his mother Döndü i..."
34884,2017,Non-Transferable,Turkish,Brendan Bradley,"YouTubers Shanna Malcolm, Shira Lazar, Sara Fl...",romantic comedy,https://en.wikipedia.org/wiki/Non-Transferable...,The film centres around a young woman named Am...


In [8]:
df = df[:1000]
df

,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot
0,1901,Kansas Saloon Smashers,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Kansas_Saloon_Sm...,"A bartender is working at a saloon, serving dr..."
1,1901,Love by the Light of the Moon,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Love_by_the_Ligh...,"The moon, painted with a smiling face hangs ov..."
2,1901,The Martyred Presidents,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/The_Martyred_Pre...,"The film, just over a minute long, is composed..."
3,1901,"Terrible Teddy, the Grizzly King",American,Unknown,NaN,unknown,"https://en.wikipedia.org/wiki/Terrible_Teddy,_...",Lasting just 61 seconds and consisting of two ...
4,1902,Jack and the Beanstalk,American,"George S. Fleming, Edwin S. Porter",NaN,unknown,https://en.wikipedia.org/wiki/Jack_and_the_Bea...,The earliest known adaptation of the classic f...
...,...,...,...,...,...,...,...,...
995,1930,Playing Around,American,Mervyn LeRoy,"Alice White, Chester Morris",drama,https://en.wikipedia.org/wiki/Playing_Around,Alice White plays the part of a working class ...
996,1930,Raffles,American,George Fitzmaurice,"Ronald Colman, Kay Francis",mystery,https://en.wikipedia.org/wiki/Raffles_(1930_film),Gentleman jewel thief A.J. Raffles (Ronald Col...
997,1930,Reaching for the Moon,American,Edmund Goulding,"Douglas Fairbanks, Edward Everett Horton, Bing...",musical,https://en.wikipedia.org/wiki/Reaching_for_the...,"Wall Street wizard, Larry Day, new to the ways..."
998,1930,Recaptured Love,American,John G. Adolfi,"Belle Bennett, Dorothy Burgess",musical,https://en.wikipedia.org/wiki/Recaptured_Love,"In this drama, a 50-year-old married man (play..."


In [9]:
from datasets import Dataset

def map_dataset(batch):
  return tokenizer(
      batch["Plot"],
      truncation=True,
      max_length=128,
      return_overflowing_tokens=True
  )


dataset = Dataset.from_pandas(df)
dataset = dataset.map(
    map_dataset,
    batched=True,
    batch_size=8,
    remove_columns=list(df.columns),
  
)
dataset = dataset.remove_columns(["overflow_to_sample_mapping"])
dataset = dataset.train_test_split(test_size=0.2)
dataset

Map: 100%|██████████| 1000/1000 [00:01<00:00, 931.29 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 2188
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 548
    })
})

In [10]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

data_collator

DataCollatorForLanguageModeling(tokenizer=GPT2TokenizerFast(name_or_path='distilgpt2', vocab_size=50257, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
), mlm=False, whole_word_mask=False, mlm_probability=0.15, mask_replace_prob=0.8, random_replace_prob=0.1, pad_to_multiple_of=None, tf_experimental_compile=False, return_tensors='pt', seed=None)

In [11]:
from transformers import TrainingArguments, Trainer, AutoModelForCausalLM



training_args = TrainingArguments(
    output_dir=".output/model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=10,
    per_device_train_batch_size=4,   
    per_device_eval_batch_size=4,
    fp16=False,                      
    dataloader_num_workers=0         
)

trainer = Trainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=training_args,
    data_collator=data_collator
)


trainer.train()

c:\xampp\htdocs\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.973000,3.728512
2,3.744500,3.694455
3,3.648700,3.686071
4,3.576000,3.679365
5,3.518600,3.681507
6,3.472700,3.683738
7,3.432400,3.685329
8,3.402400,3.691582
9,3.367000,3.692039
10,3.355000,3.693452


c:\xampp\htdocs\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\xampp\htdocs\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\xampp\htdocs\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\xampp\htdocs\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\xampp\htdocs\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument i

TrainOutput(global_step=5470, training_loss=3.5316600178887683, metrics={'train_runtime': 20958.6425, 'train_samples_per_second': 1.044, 'train_steps_per_second': 0.261, 'total_flos': 712191852380160.0, 'train_loss': 3.5316600178887683, 'epoch': 10.0})

In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained("./.output/model/checkpoint-2500")

model

prompt ="A short film about a pilot who"
inputs = tokenizer(prompt, return_tensors="pt").to("cpu")

outputs = model.generate(
    inputs.input_ids,
    max_new_tokens=100,
    do_sample=True,
    top_k=50,
    top_p=0.95
)
outputs

output_string = tokenizer.batch_decode(outputs)

output_string

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


['A short film about a pilot who finds himself trapped in a railroad track has been released to celebrate the 50th anniversary of his death. The final scene, in the film\'s original form (as described in a film magazine in 1958), depicts a strange sequence of flying machines, named "the Flying Circus", which are seen at the entrance to the amusement park. The pilot, who is apparently a pilot for a company, is actually the same pilot who is in a field job as the engineer. However, the machine "hired"']